# Pick the cube you name: a live demo

Three cubes, red, green and blue, lie on the table beside a bin in the SAPIEN twin of the SO-101 arm. Say which cube goes next, in your own words ("grab the green one and drop it in the box"), or type it. Gemma 4 E4B hears or reads the sentence and answers with the command and its fields; a program reads the one cube those fields name; SAM 3 outlines every cube in both camera pictures; the two cubes you did not name are painted over; SmolVLA, a vision-language-action model that cannot choose between objects by itself (`../README.md`, "Why SAM 3 is needed"), then sees one cube and picks it. Nothing in SmolVLA was changed or fine-tuned.

| Stage | What happens | Model or program |
|---|---|---|
| 1. Understand the command | Gemma 4 answers in one fixed shape: `pick_place`, the object words, the place words, what it heard | Gemma 4 E4B on FastFlowLM (NPU) or `llama-server` (iGPU), `backend` in `../configs/models.toml` (`so101_smolvla.voice`) |
| 2. Choose the cube | the one colour in the object field; "the cube", two colours, a question or another command are refused and nothing moves | `cube_commands.py` |
| 3. Outline the cubes | every cube in both pictures, each time SmolVLA plans (every 50 steps); the colour inside an outline says which cube it is | SAM 3, `sam3-tracker-rocm` |
| 4. Hide the other cubes | the two cubes not named are painted over with the empty table and bin | `picture_editor.py` |
| 5. Move the arm | 50 joint targets per plan, 400 steps per command; the simulator's rule decides when the cube counts as placed | SmolVLA `so101-smolvla-sim-real-10task` |

**What you see.** Six pictures, a row per camera (top, wrist): the scene as the camera takes it at every step; the outlines SAM 3 gave when SmolVLA last planned, tinted by what was done with each (target, painted over, not used); and the picture SmolVLA was given. Beside them: the voice meter (the microphone's level, green while Silero counts it as speech), the transcript (what Gemma heard, the call it made, what the arm did), the command's progress, and the gripper (the jaw opening the simulator reports, green while a jaw touches the cube). The utilization and power table opens as a small window of its own. The pick-and-place loop is the same code the automated test runs (`so101_smolvla.pick_place`), so what you see here is what the test measures.

**Needed:** `../scripts/setup_voice.sh` (Gemma, the packages), `../scripts/setup_sam3_env.sh`, `../scripts/setup_sapien_env.sh`, and this notebook's kernel ("SO-101 SmolVLA notebooks", registered by `./run_notebooks.sh`). One command takes about 30 s (400 steps); Gemma's call about 2 s after a spoken sentence on the NPU; SAM 3 0.3 s per camera per plan (`../TESTING_LEDGER.md`).

## 1. Start everything

Gemma's server, the SAM 3 worker, SmolVLA, the SAPIEN scene and the utilization window. Each part prints its seconds as it comes up. A missing piece stops the cell with its name.

In [ ]:
import os
import sys
import tomllib
from pathlib import Path

WORKSHOP = Path('..').resolve()
RIGHT_KERNEL = 'Select the kernel "SO-101 SmolVLA notebooks"; run_notebooks.sh in this directory registers it.'
try:
    from so101_smolvla.demo import DemoStack
    from so101_smolvla.demo_view import DemoView
    from so101_smolvla.sapien_config import SAPIEN_PYTHON, SAPIEN_SOURCE
    from so101_smolvla.utilization_meters import close_meters, open_meters
    from so101_smolvla.voice.settings import check_ready
except ImportError as e:
    raise RuntimeError('%s. %s' % (e, RIGHT_KERNEL)) from e
if os.path.realpath(sys.prefix) != os.path.realpath(WORKSHOP / '.venv'):
    raise RuntimeError('This kernel runs %s, not %s/.venv. %s' % (sys.prefix, WORKSHOP, RIGHT_KERNEL))

missing = check_ready()  # Gemma on the backend of ../configs/models.toml (FastFlowLM and its model, or llama-server and the model files), espeak-ng
sam3 = tomllib.loads((WORKSHOP / 'configs' / 'models.toml').read_text())['outlines']
if not (WORKSHOP / '.venv-sam3' / 'bin' / 'python').is_file() or not (
        Path(os.path.expanduser(sam3['compiled_dir'])) / 'ARTIFACT_MANIFEST.json').is_file():
    missing.append('the SAM 3 environment and its compiled model files (scripts/setup_sam3_env.sh)')
if not SAPIEN_PYTHON.is_file() or not (SAPIEN_SOURCE / '.git').is_dir():
    missing.append('the SAPIEN twin (scripts/setup_sapien_env.sh)')
if missing:
    raise RuntimeError('SETUP FAILURE: ' + '; '.join(missing))

SEED = 0  # the layout: where the cubes and the bin stand
stack = DemoStack(SEED, log_dir='logs').start()  # Gemma on its default port; log logs/flm-server.log (NPU) or logs/llama-server.log (iGPU)
meters = open_meters()
print('utilization and power table on display %s of %s (renderer %s)' % (meters['display'], meters['host'], meters['renderer']))

## 2. Compute placement

What runs where while a command is carried out (this machine: AMD Ryzen AI Max+ PRO 395, 16 cores, integrated graphics gfx1151, XDNA 2 NPU; `../README.md`, "Compute placement", has the measured processor shares).

| Task | Model or program | Runs on | Process |
|---|---|---|---|
| Microphone and VAD | 20 ms frames from the microphone; Silero speech detection (ONNX Runtime) gates each frame; a sentence goes to Gemma the moment the speaker has been silent for 0.4 s | CPU | the kernel's microphone thread (`so101_smolvla.voice.microphone`) |
| Understanding the command | Gemma 4 E4B with its audio encoder | NPU (`backend` `npu`, the default) or iGPU (`igpu`) | FastFlowLM `flm serve` (Q4_1) or `llama-server` (Q4_0, HIP), a child of the kernel |
| Choosing the cube | a program, `cube_commands.py` | CPU | the kernel's worker thread |
| Outlines | SAM 3 (`sam3-tracker-rocm`, compiled with MIGraphX) | iGPU | the SAM 3 worker (`.venv-sam3`) |
| Painting | a program, `picture_editor.py` (NumPy) | CPU | the kernel's worker thread |
| Motion | SmolVLA (`torch`, ROCm) | iGPU | the kernel's worker thread (`.venv`) |
| Digital twin | SAPIEN with PhysX: physics, both cameras, the placement rule | CPU | the SAPIEN worker (`.venv-sapien`) |
| This picture | the six tiles and the panel, drawn when something changed, at most 10 times a second | CPU | the kernel's render thread (`demo_view`) |
| Utilization and power table | the sampler and a Qt Quick window | CPU | its own process (`utilization_window.py`) |

## 3. Live

Run the cell. The picture appears with the text box and two buttons under it, and the microphone starts listening; without a microphone the panel says so and the text box is the way in. Say or type a command, for example "put the green cube in the bin". The transcript shows what Gemma heard (`you:`), the call it made (`->`), and what the arm did; a sentence that names no one cube is refused and moves nothing. While the arm works the live tiles follow every step and the other four tiles change at every plan. "new layout" rebuilds the scene with the next seed; "stop" ends the picture (the parts keep running until section 4).

A command is 400 steps. When they end with the cube still in the gripper, the command goes on until the simulator's rule says the cube is placed, and ends there. A command that ends without its cube placed, after its 400 steps with the gripper empty or at 1000 steps, has given up: a program, not SmolVLA, opens the gripper and drives the arm back to its start pose, and the transcript says so in red.

In [ ]:
view = DemoView(stack.session).show()
if stack.listen():
    print('listening to the microphone; say a command, or type one into the box')
else:
    print('no microphone could be opened; type a command into the box')

## 4. Stop

The cell shows a button, "stop everything". Pressing it ends the picture, the microphone, Gemma's server, the SAM 3 worker, the scene and the utilization window. Running the cell stops nothing, so "Run All" brings the demo up and leaves it running.

In [ ]:
from so101_smolvla.demo_view import StopControl


def stop_everything():
    view.close()
    stack.close()
    return ['table closed' if close_meters() else 'no table was running', 'Gemma, SAM 3 and the scene stopped.']


stop = StopControl(stop_everything).show()
print('the demo keeps running; press "stop everything" to end it')